In [ ]:
import pandas as pd

data = pd.read_csv('./data/daF3077_eng.csv', sep=';')
data.head()

## Traditional linear regression model

See variable descriptions in [here](https://services.fsd.tuni.fi/catalogue/FSD3077?tab=variables&study_language=en), but a quick summary:

* v4_1 I know which public welfare services and benefits are available in [country] for me and my family
* bv1 responder gender
* bv2 responder age
* bv4 ideology

Building a (poor) model to predict v4_1 from the background variables.

In [ ]:
import statsmodels.formula.api as smf

model = smf.ols('v4_1 ~ bv1 + bv2 + bv4', data=data).fit()
model.summary()

In [ ]:
print(data['v4_1'].value_counts().sort_index())

### Tasks

* Data contains also value 9 which stands for missing values. Remove it from the analysis.
* Add more data to the analysis

## The same with scikit-learn

[scikit-learn](https://scikit-learn.org/stable/) is a Python library helping with supervised machine learning.
The results ought to be about the same.
However, we do a bit different evaluations on the model, instead of just $R^2$ we also examine mean-square error (as v4_1 is continous).

In [ ]:
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

In [ ]:
features = ['bv1', 'bv2', 'bv4']
target = 'v4_1'

X = data[features]
y = data[[target]]

model = LinearRegression()
model.fit(X, y)

print('Intercept:', model.intercept_)
print('Coefficients:')

for name, coef in zip(features, model.coef_[0]):
    print(f'  {name}: {coef:.7f}')

## Model fit

There are various ways to assess models and understand how well they fit to the data, as predictive models often make errors in the analysis.
One can calculate what is the difference between predicted values by the model and the actual (or: observed) values.
Various metrics follow this kind of thinking, including

* R2
* root mean square error (RMSE)
* mean absolute error (MAE)
* area under the curve (AUC)

Different research communities have somewhat different approaches on which of these to report.

In [ ]:
predicted_values = model.predict(X)  # predicts using the model we just trained

rmse = np.sqrt(mean_squared_error(y, predicted_values))
r2   = r2_score(y, predicted_values)
mae  = mean_absolute_error(y, predicted_values)

print(f'RMSE:     {rmse:.9f}')
print(f'Rsquared: {r2:.9f}')
print(f'MAE:      {mae:.9f}')

plt.figure(figsize=(6, 6))
plt.scatter(y, predicted_values, alpha=0.3, s=10)
plt.xlabel('Observed Values')
plt.ylabel('Predicted Values')
plt.title('Observed vs Predicted Values')
plt.tight_layout()
plt.show()

### Tasks

* What does the figure above tell you about the model fit? Explain to a partner.
* Draw a plot showing the error and absolute errors per observe values

## Train-test split

Instead of using all materials for model training, let use only part of the data and then assess our performance with data not yet seen by the model.
This avoids overfit.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y
)

# Check what the data look like

print(y_train.value_counts().sort_index())
print(y_test.value_counts().sort_index())

In [ ]:
sklearn_model = LinearRegression()
sklearn_model.fit(X_train, y_train)

print('Intercept:', sklearn_model.intercept_)
print('Coefficients:')
for name, coef in zip(features, sklearn_model.coef_[0]):
    print(f'  {name}: {coef:.7f}')

In [ ]:
predicted_values = sklearn_model.predict(X_test)  # predicts using the model with unused X_test

rmse = np.sqrt(mean_squared_error(y_test, predicted_values))
r2   = r2_score(y_test, predicted_values)
mae  = mean_absolute_error(y_test, predicted_values)

print(f'RMSE:     {rmse:.9f}')
print(f'Rsquared: {r2:.9f}')
print(f'MAE:      {mae:.9f}')

### Tasks

* Try different train-test split ratios
* How much does the performance drop between train data and test data? What does that tell you?

## Cross validation

In [ ]:
from sklearn.model_selection import RepeatedKFold

# 10-fold CV repeated ten times
cv = RepeatedKFold(n_splits=10, n_repeats=10)

In [ ]:
from sklearn.model_selection import cross_validate

cv_model = LinearRegression()

cv_results = cross_validate(
    cv_model, X_train, y_train, cv=cv,
    scoring=['neg_root_mean_squared_error', 'r2', 'neg_mean_absolute_error'],
    return_train_score=False
)

print('Cross-validation results (mean over all folds):')
print(f'  RMSE:     {-cv_results["test_neg_root_mean_squared_error"].mean():.9f}')
print(f'  Rsquared: {cv_results["test_r2"].mean():.9f}')
print(f'  MAE:      {-cv_results["test_neg_mean_absolute_error"].mean():.9f}')

In [ ]:
# Fit final model on entire train set and evaluate on held-out test set
cv_model.fit(X_train, y_train)
predicted_values = cv_model.predict(X_test)  # predicts using the model with unused X_test

rmse = np.sqrt(mean_squared_error(y_test, predicted_values))
r2   = r2_score(y_test, predicted_values)
mae  = mean_absolute_error(y_test, predicted_values)

print(f'RMSE:     {rmse:.9f}')
print(f'Rsquared: {r2:.9f}')
print(f'MAE:      {mae:.9f}')

### Tasks

* Try different cross-validation rations?
* How much does the performance increase compared with non-cross-validated model?